In [1]:
%%writefile advanced_gas_prediction.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, TimeSeriesSplit, cross_val_score,
                                   validation_curve, learning_curve, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import joblib
import datetime
import warnings
import os
from scipy import stats
from scipy.stats import pearsonr
import itertools
warnings.filterwarnings("ignore")

# Try to import XGBoost, fall back if not available
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available, using RandomForest as default")
    XGBOOST_AVAILABLE = False

class AdvancedGasUsagePredictionModel:
    def __init__(self, data_path='data/data.csv'):
        """Initialize the advanced gas usage prediction model with comprehensive overfitting detection."""
        self.data_path = data_path
        self.model = None
        self.scaler = StandardScaler()
        self.features = None
        self.data = None
        self.overfitting_results = {}

    def load_data(self):
        """Load and preprocess the data."""
        # Load the CSV data
        df = pd.read_csv(self.data_path)

        # Convert timestamp to datetime
        df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M')

        # Create time-based features
        df['hour'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.dayofweek
        df['day_of_month'] = df['timestamp'].dt.day
        df['month'] = df['timestamp'].dt.month
        df['year'] = df['timestamp'].dt.year

        # Convert hour to cyclical features to represent time's cyclic nature
        df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

        # Add day of week as cyclical features
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)

        # Sort by timestamp to maintain time order
        df = df.sort_values('timestamp')

        # Calculate lag features (previous hour, day, week)
        df['hourly_volume_lag1'] = df['hourly_volume'].shift(1)
        df['hourly_volume_lag24'] = df['hourly_volume'].shift(24)  # Previous day, same hour
        df['hourly_volume_lag168'] = df['hourly_volume'].shift(168)  # Previous week, same hour

        # Calculate rolling averages
        df['hourly_volume_rolling_mean_24h'] = df['hourly_volume'].rolling(window=24).mean()
        df['hourly_volume_rolling_mean_7d'] = df['hourly_volume'].rolling(window=168).mean()

        # Drop rows with NaN values from lag creation
        df = df.dropna()

        # Store a copy of the original data
        self.data = df.copy()

        return df

    def prepare_features(self, df=None):
        """Prepare features for model training."""
        if df is None:
            df = self.load_data()

        # Features selection
        self.features = [
            'density', 'pressure_diff', 'pressure', 'temperature',
            'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos',
            'day_of_month', 'month',
            'hourly_volume_lag1', 'hourly_volume_lag24', 'hourly_volume_lag168',
            'hourly_volume_rolling_mean_24h', 'hourly_volume_rolling_mean_7d'
        ]

        # Target variable
        target = 'hourly_volume'

        # Split features and target
        X = df[self.features]
        y = df[target]

        return X, y

    def detect_data_leakage(self, X, y):
        """Advanced Method 1: Detect potential data leakage."""
        print("🔍 ADVANCED METHOD 1: DATA LEAKAGE DETECTION")
        print("="*60)

        leakage_detected = False

        # Check correlations between features and target
        correlations = {}
        for feature in self.features:
            if feature in X.columns:
                corr, p_value = pearsonr(X[feature], y)
                correlations[feature] = {'correlation': corr, 'p_value': p_value}

        # Sort by absolute correlation
        sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]['correlation']), reverse=True)

        print("📊 FEATURE-TARGET CORRELATIONS:")
        for feature, stats in sorted_corr[:10]:
            corr = stats['correlation']
            p_val = stats['p_value']

            if abs(corr) > 0.95:
                print(f"🚨 {feature}: {corr:.4f} (p={p_val:.4f}) - POTENTIAL LEAKAGE!")
                leakage_detected = True
            elif abs(corr) > 0.8:
                print(f"⚠️  {feature}: {corr:.4f} (p={p_val:.4f}) - High correlation")
            else:
                print(f"✅ {feature}: {corr:.4f} (p={p_val:.4f})")

        # Check for near-perfect linear combinations
        from sklearn.linear_model import LinearRegression
        lr_temp = LinearRegression()

        for feature in self.features:
            if feature in X.columns and 'lag' not in feature:
                other_features = [f for f in self.features if f != feature and f in X.columns]
                if len(other_features) > 0:
                    try:
                        lr_temp.fit(X[other_features], X[feature])
                        pred = lr_temp.predict(X[other_features])
                        r2_feature = r2_score(X[feature], pred)

                        if r2_feature > 0.95:
                            print(f"🚨 {feature} can be predicted from other features with R²={r2_feature:.4f}")
                            leakage_detected = True
                    except:
                        pass

        self.overfitting_results['data_leakage'] = {
            'detected': leakage_detected,
            'correlations': correlations
        }

        return leakage_detected

    def time_series_cross_validation_advanced(self, X, y, models_dict):
        """Advanced Method 2: Comprehensive Time Series Cross-Validation."""
        print("\n🔍 ADVANCED METHOD 2: COMPREHENSIVE TIME SERIES CV")
        print("="*60)

        # Different CV strategies
        cv_strategies = {
            'TimeSeriesSplit_3': TimeSeriesSplit(n_splits=3),
            'TimeSeriesSplit_5': TimeSeriesSplit(n_splits=5),
            'TimeSeriesSplit_10': TimeSeriesSplit(n_splits=10)
        }

        results = {}

        for cv_name, cv_strategy in cv_strategies.items():
            print(f"\n📈 Testing with {cv_name}:")
            results[cv_name] = {}

            for model_name, model in models_dict.items():
                scores = cross_val_score(model, X, y, cv=cv_strategy,
                                       scoring='r2', n_jobs=-1)

                mean_score = np.mean(scores)
                std_score = np.std(scores)

                results[cv_name][model_name] = {
                    'mean_r2': mean_score,
                    'std_r2': std_score,
                    'scores': scores
                }

                print(f"  {model_name}: R² = {mean_score:.4f} (±{std_score:.4f})")

                # Check for high variance (instability)
                if std_score > 0.1:
                    print(f"    ⚠️  HIGH VARIANCE - Unstable performance")
                elif std_score > 0.05:
                    print(f"    ⚠️  MODERATE VARIANCE")
                else:
                    print(f"    ✅ LOW VARIANCE - Stable performance")

        self.overfitting_results['time_series_cv'] = results
        return results

    def walk_forward_validation(self, X, y, models_dict, window_size=0.7):
        """Advanced Method 3: Walk-Forward Validation for Time Series."""
        print("\n🔍 ADVANCED METHOD 3: WALK-FORWARD VALIDATION")
        print("="*60)

        n_samples = len(X)
        initial_train_size = int(n_samples * window_size)
        results = {}

        for model_name, model in models_dict.items():
            print(f"\n🚀 Testing {model_name}:")

            predictions = []
            actuals = []
            train_scores = []
            test_scores = []

            # Walk forward through time
            for i in range(initial_train_size, n_samples - 1):
                # Train on all data up to point i
                X_train = X.iloc[:i]
                y_train = y.iloc[:i]

                # Test on next point
                X_test = X.iloc[i:i+1]
                y_test = y.iloc[i:i+1]

                # Fit model
                model_copy = type(model)(**model.get_params() if hasattr(model, 'get_params') else {})
                model_copy.fit(X_train, y_train)

                # Predict
                y_pred = model_copy.predict(X_test)[0]
                y_actual = y_test.iloc[0]

                predictions.append(y_pred)
                actuals.append(y_actual)

                # Track training performance
                train_pred = model_copy.predict(X_train)
                train_r2 = r2_score(y_train, train_pred)
                test_r2 = r2_score([y_actual], [y_pred])

                train_scores.append(train_r2)
                test_scores.append(test_r2)

            # Calculate overall metrics
            overall_r2 = r2_score(actuals, predictions)
            overall_rmse = np.sqrt(mean_squared_error(actuals, predictions))

            # Calculate performance degradation over time
            avg_train_r2 = np.mean(train_scores)
            avg_test_r2 = np.mean(test_scores)
            performance_gap = avg_train_r2 - avg_test_r2

            results[model_name] = {
                'overall_r2': overall_r2,
                'overall_rmse': overall_rmse,
                'avg_train_r2': avg_train_r2,
                'avg_test_r2': avg_test_r2,
                'performance_gap': performance_gap,
                'predictions': predictions,
                'actuals': actuals
            }

            print(f"  Overall R²: {overall_r2:.4f}")
            print(f"  Overall RMSE: {overall_rmse:.4f}")
            print(f"  Avg Train R²: {avg_train_r2:.4f}")
            print(f"  Avg Test R²: {avg_test_r2:.4f}")
            print(f"  Performance Gap: {performance_gap:.4f}")

            if performance_gap > 0.1:
                print(f"    🚨 HIGH GAP - Strong overfitting signal")
            elif performance_gap > 0.05:
                print(f"    ⚠️  MODERATE GAP - Some overfitting")
            else:
                print(f"    ✅ LOW GAP - Good generalization")

        self.overfitting_results['walk_forward'] = results
        return results

    def nested_cross_validation(self, X, y):
        """Advanced Method 4: Nested Cross-Validation for unbiased model selection."""
        print("\n🔍 ADVANCED METHOD 4: NESTED CROSS-VALIDATION")
        print("="*60)

        # Outer CV for final performance estimation
        outer_cv = TimeSeriesSplit(n_splits=3)

        # Inner CV for hyperparameter tuning
        inner_cv = TimeSeriesSplit(n_splits=3)

        # Models with hyperparameter grids
        models_params = {
            'Ridge': {
                'model': Ridge(),
                'params': {'alpha': [0.1, 1.0, 10.0, 100.0]}
            },
            'Random Forest': {
                'model': RandomForestRegressor(random_state=42),
                'params': {'n_estimators': [50, 100], 'max_depth': [5, 10, 15]}
            }
        }

        nested_results = {}

        for model_name, config in models_params.items():
            print(f"\n🔬 Nested CV for {model_name}:")

            outer_scores = []
            best_params_list = []

            for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
                # Split data
                X_train_outer = X.iloc[train_idx]
                X_test_outer = X.iloc[test_idx]
                y_train_outer = y.iloc[train_idx]
                y_test_outer = y.iloc[test_idx]

                # Inner CV for hyperparameter tuning
                grid_search = GridSearchCV(
                    config['model'], config['params'],
                    cv=inner_cv, scoring='r2', n_jobs=-1
                )

                grid_search.fit(X_train_outer, y_train_outer)

                # Test best model on outer test set
                best_score = grid_search.score(X_test_outer, y_test_outer)
                outer_scores.append(best_score)
                best_params_list.append(grid_search.best_params_)

                print(f"  Fold {fold+1}: R² = {best_score:.4f}, Best params: {grid_search.best_params_}")

            # Final nested CV score
            nested_score = np.mean(outer_scores)
            nested_std = np.std(outer_scores)

            nested_results[model_name] = {
                'nested_score': nested_score,
                'nested_std': nested_std,
                'outer_scores': outer_scores,
                'best_params_list': best_params_list
            }

            print(f"  📊 Nested CV Score: {nested_score:.4f} (±{nested_std:.4f})")

        self.overfitting_results['nested_cv'] = nested_results
        return nested_results

    def bootstrap_validation(self, X, y, models_dict, n_bootstrap=100):
        """Advanced Method 5: Bootstrap Validation."""
        print("\n🔍 ADVANCED METHOD 5: BOOTSTRAP VALIDATION")
        print("="*60)

        bootstrap_results = {}

        for model_name, model in models_dict.items():
            print(f"\n🎲 Bootstrap validation for {model_name}:")

            bootstrap_scores = []

            for i in range(n_bootstrap):
                # Bootstrap sample
                indices = np.random.choice(len(X), size=len(X), replace=True)
                X_boot = X.iloc[indices]
                y_boot = y.iloc[indices]

                # Out-of-bag sample
                oob_indices = np.setdiff1d(np.arange(len(X)), indices)
                if len(oob_indices) > 0:
                    X_oob = X.iloc[oob_indices]
                    y_oob = y.iloc[oob_indices]

                    # Fit and predict
                    model_copy = type(model)(**model.get_params() if hasattr(model, 'get_params') else {})
                    model_copy.fit(X_boot, y_boot)
                    y_pred = model_copy.predict(X_oob)

                    # Score
                    score = r2_score(y_oob, y_pred)
                    bootstrap_scores.append(score)

            # Calculate confidence intervals
            scores_array = np.array(bootstrap_scores)
            mean_score = np.mean(scores_array)
            std_score = np.std(scores_array)
            ci_lower = np.percentile(scores_array, 2.5)
            ci_upper = np.percentile(scores_array, 97.5)

            bootstrap_results[model_name] = {
                'mean_score': mean_score,
                'std_score': std_score,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'scores': bootstrap_scores
            }

            print(f"  Mean R²: {mean_score:.4f} (±{std_score:.4f})")
            print(f"  95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

        self.overfitting_results['bootstrap'] = bootstrap_results
        return bootstrap_results

    def learning_curves_comprehensive(self, X, y, models_dict):
        """Advanced Method 6: Comprehensive Learning Curves Analysis."""
        print("\n🔍 ADVANCED METHOD 6: COMPREHENSIVE LEARNING CURVES")
        print("="*60)

        train_sizes = np.linspace(0.1, 1.0, 10)
        learning_results = {}

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()

        for idx, (model_name, model) in enumerate(models_dict.items()):
            if idx >= 4:  # Only plot first 4 models
                break

            print(f"\n📈 Learning curves for {model_name}:")

            train_sizes_abs, train_scores, val_scores = learning_curve(
                model, X, y, train_sizes=train_sizes,
                cv=TimeSeriesSplit(n_splits=3), scoring='r2', n_jobs=-1
            )

            # Calculate statistics
            train_mean = np.mean(train_scores, axis=1)
            train_std = np.std(train_scores, axis=1)
            val_mean = np.mean(val_scores, axis=1)
            val_std = np.std(val_scores, axis=1)

            # Check for overfitting patterns
            final_gap = train_mean[-1] - val_mean[-1]
            gap_trend = np.polyfit(range(len(train_mean)), train_mean - val_mean, 1)[0]

            learning_results[model_name] = {
                'train_sizes': train_sizes_abs,
                'train_mean': train_mean,
                'val_mean': val_mean,
                'final_gap': final_gap,
                'gap_trend': gap_trend
            }

            # Plot
            axes[idx].plot(train_sizes_abs, train_mean, 'o-', label='Training')
            axes[idx].fill_between(train_sizes_abs, train_mean - train_std,
                                 train_mean + train_std, alpha=0.1)

            axes[idx].plot(train_sizes_abs, val_mean, 's-', label='Validation')
            axes[idx].fill_between(train_sizes_abs, val_mean - val_std,
                                 val_mean + val_std, alpha=0.1)

            axes[idx].set_title(f'{model_name}')
            axes[idx].set_xlabel('Training Size')
            axes[idx].set_ylabel('R² Score')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)

            print(f"  Final gap: {final_gap:.4f}")
            print(f"  Gap trend: {gap_trend:.6f} (negative = converging)")

            if final_gap > 0.1:
                print(f"    🚨 LARGE GAP - Strong overfitting")
            elif final_gap > 0.05:
                print(f"    ⚠️  MODERATE GAP - Some overfitting")
            else:
                print(f"    ✅ SMALL GAP - Good generalization")

        plt.tight_layout()
        plt.show()

        self.overfitting_results['learning_curves'] = learning_results
        return learning_results

    def residual_analysis_advanced(self, X, y, models_dict):
        """Advanced Method 7: Comprehensive Residual Analysis."""
        print("\n🔍 ADVANCED METHOD 7: ADVANCED RESIDUAL ANALYSIS")
        print("="*60)

        # Split data
        split_point = int(len(X) * 0.8)
        X_train, X_test = X[:split_point], X[split_point:]
        y_train, y_test = y[:split_point], y[split_point:]

        residual_results = {}

        fig, axes = plt.subplots(len(models_dict), 4, figsize=(20, 5*len(models_dict)))
        if len(models_dict) == 1:
            axes = axes.reshape(1, -1)

        for idx, (model_name, model) in enumerate(models_dict.items()):
            print(f"\n🔬 Residual analysis for {model_name}:")

            # Fit model
            model.fit(X_train, y_train)

            # Predictions
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)

            # Residuals
            residuals_train = y_train - y_pred_train
            residuals_test = y_test.values - y_pred_test

            # Statistical tests
            # 1. Normality test (Shapiro-Wilk)
            shapiro_stat, shapiro_p = stats.shapiro(residuals_test[:min(5000, len(residuals_test))])

            # 2. Autocorrelation test (Durbin-Watson)
            from statsmodels.stats.diagnostic import acorr_ljungbox
            ljung_box_stat, ljung_box_p = acorr_ljungbox(residuals_test, lags=10, return_df=False)

            # 3. Heteroscedasticity test (White test)
            from sklearn.linear_model import LinearRegression
            lr_temp = LinearRegression()
            try:
                # Regress squared residuals on predictions
                residuals_squared = residuals_test ** 2
                lr_temp.fit(y_pred_test.reshape(-1, 1), residuals_squared)
                pred_sq_res = lr_temp.predict(y_pred_test.reshape(-1, 1))
                white_stat = len(residuals_test) * r2_score(residuals_squared, pred_sq_res)
                white_p = 1 - stats.chi2.cdf(white_stat, df=1)
            except:
                white_stat, white_p = np.nan, np.nan

            residual_results[model_name] = {
                'shapiro_stat': shapiro_stat,
                'shapiro_p': shapiro_p,
                'ljung_box_stat': ljung_box_stat,
                'ljung_box_p': ljung_box_p,
                'white_stat': white_stat,
                'white_p': white_p,
                'residuals_mean': np.mean(residuals_test),
                'residuals_std': np.std(residuals_test)
            }

            # Plotting
            # 1. Residuals vs Fitted
            axes[idx, 0].scatter(y_pred_test, residuals_test, alpha=0.6)
            axes[idx, 0].axhline(y=0, color='r', linestyle='--')
            axes[idx, 0].set_xlabel('Fitted Values')
            axes[idx, 0].set_ylabel('Residuals')
            axes[idx, 0].set_title(f'{model_name}: Residuals vs Fitted')
            axes[idx, 0].grid(True, alpha=0.3)

            # 2. Q-Q plot
            stats.probplot(residuals_test, dist="norm", plot=axes[idx, 1])
            axes[idx, 1].set_title(f'{model_name}: Q-Q Plot')

            # 3. Histogram of residuals
            axes[idx, 2].hist(residuals_test, bins=30, alpha=0.7, edgecolor='black')
            axes[idx, 2].set_xlabel('Residuals')
            axes[idx, 2].set_ylabel('Frequency')
            axes[idx, 2].set_title(f'{model_name}: Residuals Distribution')
            axes[idx, 2].grid(True, alpha=0.3)

            # 4. Residuals over time
            axes[idx, 3].plot(residuals_test, alpha=0.7)
            axes[idx, 3].axhline(y=0, color='r', linestyle='--')
            axes[idx, 3].set_xlabel('Time Index')
            axes[idx, 3].set_ylabel('Residuals')
            axes[idx, 3].set_title(f'{model_name}: Residuals Over Time')
            axes[idx, 3].grid(True, alpha=0.3)

            # Statistical interpretation
            print(f"  Normality (Shapiro): p = {shapiro_p:.4f} {'✅' if shapiro_p > 0.05 else '❌'}")
            print(f"  Autocorrelation (Ljung-Box): p = {ljung_box_p:.4f} {'✅' if ljung_box_p > 0.05 else '❌'}")
            print(f"  Homoscedasticity (White): p = {white_p:.4f} {'✅' if white_p > 0.05 else '❌'}")

        plt.tight_layout()
        plt.show()

        self.overfitting_results['residual_analysis'] = residual_results
        return residual_results

    def feature_importance_stability(self, X, y, models_dict):
        """Advanced Method 8: Feature Importance Stability Analysis."""
        print("\n🔍 ADVANCED METHOD 8: FEATURE IMPORTANCE STABILITY")
        print("="*60)

        stability_results = {}

        # Only analyze tree-based models that have feature importance
        tree_models = {name: model for name, model in models_dict.items()
                      if hasattr(model, 'feature_importances_') or
                      type(model).__name__ in ['RandomForestRegressor', 'GradientBoostingRegressor', 'XGBRegressor']}

        if not tree_models:
            print("❌ No tree-based models found for feature importance analysis")
            return {}

        cv = TimeSeriesSplit(n_splits=5)

        for model_name, model in tree_models.items():
            print(f"\n🌳 Feature importance stability for {model_name}:")

            importance_matrix = []

            for fold, (train_idx, val_idx) in enumerate(cv.split(X)):
                X_train_fold = X.iloc[train_idx]
                y_train_fold = y.iloc[train_idx]

                # Fit model
                model_copy = type(model)(**model.get_params())
                model_copy.fit(X_train_fold, y_train_fold)

                # Get feature importance
                if hasattr(model_copy, 'feature_importances_'):
                    importance_matrix.append(model_copy.feature_importances_)

            if importance_matrix:
                importance_matrix = np.array(importance_matrix)

                # Calculate stability metrics
                mean_importance = np.mean(importance_matrix, axis=0)
                std_importance = np.std(importance_matrix, axis=0)
                cv_importance = std_importance / (mean_importance + 1e-8)  # Coefficient of variation

                # Create feature importance DataFrame
                importance_df = pd.DataFrame({
                    'Feature': self.features,
                    'Mean_Importance': mean_importance,
                    'Std_Importance': std_importance,
                    'CV_Importance': cv_importance
                }).sort_values('Mean_Importance', ascending=False)

                stability_results[model_name] = importance_df

                print("  📊 Top 5 Most Important Features:")
                for i, (_, row) in enumerate(importance_df.head().iterrows()):
                    stability_status = "✅ Stable" if row['CV_Importance'] < 0.3 else "⚠️ Unstable"
                    print(f"    {i+1}. {row['Feature']}: {row['Mean_Importance']:.4f} "
                          f"(CV: {row['CV_Importance']:.3f}) {stability_status}")

        # Plot feature importance stability
        if stability_results:
            fig, axes = plt.subplots(len(stability_results), 1, figsize=(12, 6*len(stability_results)))
            if len(stability_results) == 1:
                axes = [axes]

            for idx, (model_name, importance_df) in enumerate(stability_results.items()):
                top_features = importance_df.head(10)

                bars = axes[idx].bar(range(len(top_features)), top_features['Mean_Importance'],
                                   yerr=top_features['Std_Importance'], capsize=3)
                axes[idx].set_xlabel('Features')
                axes[idx].set_ylabel('Importance')
                axes[idx].set_title(f'{model_name}: Feature Importance Stability')
                axes[idx].set_xticks(range(len(top_features)))
                axes[idx].set_xticklabels(top_features['Feature'], rotation=45, ha='right')
                axes[idx].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

        self.overfitting_results['feature_stability'] = stability_results
        return stability_results

    def comprehensive_overfitting_report(self):
        """Generate comprehensive overfitting assessment report."""
        print("\n" + "="*80)
        print("🎯 COMPREHENSIVE OVERFITTING ASSESSMENT REPORT")
        print("="*80)

        # Prepare data
        df = self.load_data()
        X, y = self.prepare_features(df)

        # Scale features
        X_scaled = pd.DataFrame(self.scaler.fit_transform(X),
                               columns=X.columns, index=X.index)

        # Define models to test
        models_dict = {
            'Linear Regression': LinearRegression(),
            'Ridge (alpha=1.0)': Ridge(alpha=1.0),
            'Ridge (alpha=10.0)': Ridge(alpha=10.0),
            'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                                          max_depth=5, random_state=42)
        }

        if XGBOOST_AVAILABLE:
            models_dict['XGBoost'] = XGBRegressor(n_estimators=100, learning_rate=0.1,
                                                max_depth=5, random_state=42)

        print(f"📊 Analyzing {len(models_dict)} models with {len(X)} samples and {len(self.features)} features")

        # Run all advanced overfitting detection methods
        self.detect_data_leakage(X_scaled, y)
        self.time_series_cross_validation_advanced(X_scaled, y, models_dict)
        self.walk_forward_validation(X_scaled, y, models_dict)
        self.nested_cross_validation(X_scaled, y)
        self.bootstrap_validation(X_scaled, y, models_dict)
        self.learning_curves_comprehensive(X_scaled, y, models_dict)
        self.residual_analysis_advanced(X_scaled, y, models_dict)
        self.feature_importance_stability(X_scaled, y, models_dict)

        # Final comprehensive assessment
        print("\n" + "="*80)
        print("🏆 FINAL COMPREHENSIVE ASSESSMENT")
        print("="*80)

        risk_score = 0
        max_risk_score = 8

        # Check 1: Data leakage
        if self.overfitting_results.get('data_leakage', {}).get('detected', False):
            risk_score += 2
            print("❌ Data leakage detected (+2 points)")
        else:
            print("✅ No data leakage detected (0 points)")

        # Check 2: Cross-validation stability
        cv_results = self.overfitting_results.get('time_series_cv', {})
        high_variance_count = 0
        for cv_name, cv_result in cv_results.items():
            for model_name, result in cv_result.items():
                if result['std_r2'] > 0.1:
                    high_variance_count += 1

        if high_variance_count > len(models_dict) * 0.5:
            risk_score += 1
            print("❌ High variance in cross-validation (+1 point)")
        else:
            print("✅ Stable cross-validation performance (0 points)")

        # Check 3: Walk-forward validation gaps
        wf_results = self.overfitting_results.get('walk_forward', {})
        high_gap_count = 0
        for model_name, result in wf_results.items():
            if result['performance_gap'] > 0.1:
                high_gap_count += 1

        if high_gap_count > 0:
            risk_score += 2
            print("❌ High performance gaps in walk-forward validation (+2 points)")
        elif high_gap_count > len(models_dict) * 0.3:
            risk_score += 1
            print("⚠️  Some performance gaps in walk-forward validation (+1 point)")
        else:
            print("✅ Good walk-forward validation performance (0 points)")

        # Check 4: Learning curves
        lc_results = self.overfitting_results.get('learning_curves', {})
        high_gap_lc_count = 0
        for model_name, result in lc_results.items():
            if result['final_gap'] > 0.1:
                high_gap_lc_count += 1

        if high_gap_lc_count > len(models_dict) * 0.5:
            risk_score += 1
            print("❌ Learning curves show overfitting (+1 point)")
        else:
            print("✅ Learning curves look good (0 points)")

        # Check 5: Residual analysis
        residual_results = self.overfitting_results.get('residual_analysis', {})
        residual_issues = 0
        for model_name, result in residual_results.items():
            if result['shapiro_p'] < 0.05:  # Non-normal residuals
                residual_issues += 1
            if result['ljung_box_p'] < 0.05:  # Autocorrelated residuals
                residual_issues += 1

        if residual_issues > len(models_dict):
            risk_score += 1
            print("❌ Residual analysis shows model inadequacy (+1 point)")
        else:
            print("✅ Residual analysis looks acceptable (0 points)")

        # Check 6: Feature importance stability
        stability_results = self.overfitting_results.get('feature_stability', {})
        unstable_features = 0
        for model_name, importance_df in stability_results.items():
            unstable_count = (importance_df['CV_Importance'] > 0.5).sum()
            unstable_features += unstable_count

        if unstable_features > len(self.features) * 0.3:
            risk_score += 1
            print("❌ Feature importance is unstable (+1 point)")
        else:
            print("✅ Feature importance is stable (0 points)")

        # Final verdict
        risk_percentage = (risk_score / max_risk_score) * 100

        print(f"\n🎯 OVERFITTING RISK SCORE: {risk_score}/{max_risk_score} ({risk_percentage:.1f}%)")

        if risk_percentage >= 75:
            verdict = "🚨 SEVERE OVERFITTING RISK"
            recommendations = [
                "Use regularization (Ridge/Lasso with higher alpha)",
                "Remove lag features or reduce their number",
                "Collect more diverse training data",
                "Use simpler models",
                "Apply feature selection techniques",
                "Consider ensemble methods with different base models"
            ]
        elif risk_percentage >= 50:
            verdict = "⚠️  HIGH OVERFITTING RISK"
            recommendations = [
                "Add mild regularization",
                "Use more conservative cross-validation",
                "Monitor performance on completely new data",
                "Consider reducing model complexity",
                "Validate predictions manually on known cases"
            ]
        elif risk_percentage >= 25:
            verdict = "⚠️  MODERATE OVERFITTING RISK"
            recommendations = [
                "Continue monitoring with new data",
                "Consider light regularization",
                "Validate on recent unseen data",
                "Document model limitations clearly"
            ]
        else:
            verdict = "✅ LOW OVERFITTING RISK"
            recommendations = [
                "Model appears to generalize well",
                "Continue monitoring with production data",
                "Regular retraining with new data",
                "Document model performance benchmarks"
            ]

        print(f"\n{verdict}")
        print(f"\n💡 ACTIONABLE RECOMMENDATIONS:")
        for i, rec in enumerate(recommendations, 1):
            print(f"   {i}. {rec}")

        # Save results
        self.overfitting_results['final_assessment'] = {
            'risk_score': risk_score,
            'risk_percentage': risk_percentage,
            'verdict': verdict,
            'recommendations': recommendations
        }

        return self.overfitting_results

    def train_model_with_overfitting_protection(self, model_type='ridge', alpha=1.0):
        """Train model with built-in overfitting protection based on analysis."""
        print(f"\n🛡️  TRAINING MODEL WITH OVERFITTING PROTECTION")
        print("="*60)

        # Load and prepare data
        df = self.load_data()
        X, y = self.prepare_features(df)
        X_scaled = self.scaler.fit_transform(X)

        # Select model with regularization
        if model_type == 'ridge':
            self.model = Ridge(alpha=alpha)
            print(f"Using Ridge Regression with alpha={alpha}")
        elif model_type == 'lasso':
            self.model = Lasso(alpha=alpha)
            print(f"Using Lasso Regression with alpha={alpha}")
        elif model_type == 'elastic_net':
            self.model = ElasticNet(alpha=alpha, l1_ratio=0.5)
            print(f"Using Elastic Net with alpha={alpha}")
        elif model_type == 'random_forest':
            self.model = RandomForestRegressor(
                n_estimators=50, max_depth=10, min_samples_split=10,
                min_samples_leaf=5, random_state=42
            )
            print("Using Random Forest with conservative parameters")
        else:
            self.model = LinearRegression()
            print("Using Linear Regression")

        # Time series split for proper validation
        tscv = TimeSeriesSplit(n_splits=5)
        scores = cross_val_score(self.model, X_scaled, y, cv=tscv, scoring='r2')

        print(f"Cross-validation R² scores: {scores}")
        print(f"Mean CV R²: {np.mean(scores):.4f} (±{np.std(scores):.4f})")

        # Train final model on all data
        self.model.fit(X_scaled, y)

        # Make predictions for evaluation
        y_pred = self.model.predict(X_scaled)
        train_r2 = r2_score(y, y_pred)
        train_rmse = np.sqrt(mean_squared_error(y, y_pred))

        print(f"Final model training R²: {train_r2:.4f}")
        print(f"Final model training RMSE: {train_rmse:.4f}")
        print(f"Gap between training and CV: {train_r2 - np.mean(scores):.4f}")

        return {
            'model': self.model,
            'cv_scores': scores,
            'train_r2': train_r2,
            'train_rmse': train_rmse
        }

    def save_model(self, model_path='models/gas_usage_model.pkl'):
        """Save the trained model and all analysis results."""
        if self.model is None:
            raise ValueError("Model has not been trained yet")

        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(model_path), exist_ok=True)

        # Save model, scaler, features, and overfitting analysis
        joblib.dump({
            'model': self.model,
            'scaler': self.scaler,
            'features': self.features,
            'overfitting_results': self.overfitting_results
        }, model_path)

        print(f"✅ Model and analysis results saved to {model_path}")

# Example usage
if __name__ == "__main__":
    # Initialize advanced model
    advanced_model = AdvancedGasUsagePredictionModel(data_path='data/data.csv')

    # Run comprehensive overfitting analysis
    results = advanced_model.comprehensive_overfitting_report()

    # Train model with overfitting protection
    model_results = advanced_model.train_model_with_overfitting_protection(
        model_type='ridge', alpha=10.0
    )

    # Save everything
    advanced_model.save_model()

Writing advanced_gas_prediction.py


In [ ]:
"""
Run Advanced Gas Prediction with Comprehensive Overfitting Detection

This script runs the most advanced overfitting detection methods on your gas usage data.
"""

from advanced_gas_prediction import AdvancedGasUsagePredictionModel
import warnings
warnings.filterwarnings("ignore")

def main():
    print("🚀 STARTING ADVANCED GAS USAGE ANALYSIS")
    print("="*80)

    # Initialize the advanced model
    model = AdvancedGasUsagePredictionModel(data_path='data.csv')

    print("📋 This analysis will run 8 advanced overfitting detection methods:")
    print("   1. Data Leakage Detection")
    print("   2. Time Series Cross-Validation (3 strategies)")
    print("   3. Walk-Forward Validation")
    print("   4. Nested Cross-Validation")
    print("   5. Bootstrap Validation")
    print("   6. Learning Curves Analysis")
    print("   7. Advanced Residual Analysis")
    print("   8. Feature Importance Stability")
    print("\n⏱️  This may take 10-15 minutes to complete...\n")

    try:
        # Run comprehensive analysis
        results = model.comprehensive_overfitting_report()

        print("\n🎉 ANALYSIS COMPLETE!")
        print("="*50)

        # Display final assessment
        final_assessment = results.get('final_assessment', {})
        print(f"Risk Score: {final_assessment.get('risk_score', 'N/A')}/8")
        print(f"Risk Level: {final_assessment.get('risk_percentage', 'N/A')}%")
        print(f"Verdict: {final_assessment.get('verdict', 'N/A')}")

        # Train protected model based on results
        print("\n🛡️  Training model with overfitting protection...")

        risk_percentage = final_assessment.get('risk_percentage', 0)

        if risk_percentage >= 75:
            # High risk - use strong regularization
            model_results = model.train_model_with_overfitting_protection(
                model_type='ridge', alpha=100.0
            )
            print("✅ Used Ridge regression with strong regularization (alpha=100)")
        elif risk_percentage >= 50:
            # Moderate risk - use moderate regularization
            model_results = model.train_model_with_overfitting_protection(
                model_type='ridge', alpha=10.0
            )
            print("✅ Used Ridge regression with moderate regularization (alpha=10)")
        elif risk_percentage >= 25:
            # Low-moderate risk - use light regularization
            model_results = model.train_model_with_overfitting_protection(
                model_type='ridge', alpha=1.0
            )
            print("✅ Used Ridge regression with light regularization (alpha=1)")
        else:
            # Very low risk - minimal regularization
            model_results = model.train_model_with_overfitting_protection(
                model_type='ridge', alpha=0.1
            )
            print("✅ Used Ridge regression with minimal regularization (alpha=0.1)")

        # Save the model and results
        model.save_model('models/advanced_gas_model.pkl')

        print(f"\n📊 FINAL MODEL PERFORMANCE:")
        print(f"Cross-Validation R²: {model_results['cv_scores'].mean():.4f} (±{model_results['cv_scores'].std():.4f})")
        print(f"Training R²: {model_results['train_r2']:.4f}")
        print(f"Training RMSE: {model_results['train_rmse']:.4f}")

        # Summary recommendations
        print(f"\n💡 KEY RECOMMENDATIONS:")
        recommendations = final_assessment.get('recommendations', [])
        for i, rec in enumerate(recommendations[:3], 1):  # Show top 3
            print(f"   {i}. {rec}")

        print("\n✅ Analysis complete! Check the generated plots and saved model.")

        return results, model_results

    except Exception as e:
        print(f"❌ Error during analysis: {e}")
        print("Please check your data file path and format.")
        return None, None

if __name__ == "__main__":
    results, model_results = main()

    if results:
        print("\n🎯 QUICK OVERFITTING CHECK:")
        risk_score = results['final_assessment']['risk_score']

        if risk_score >= 6:
            print("🚨 SEVERE OVERFITTING - Your original 99.5% accuracy is likely overfitted")
        elif risk_score >= 4:
            print("⚠️  HIGH OVERFITTING RISK - Be very cautious with the model")
        elif risk_score >= 2:
            print("⚠️  MODERATE RISK - Monitor performance closely")
        else:
            print("✅ LOW RISK - Model appears reasonable")

        print(f"\nDetailed results saved in: models/advanced_gas_model.pkl")
        print("All plots and analysis have been displayed above.")

🚀 STARTING ADVANCED GAS USAGE ANALYSIS
📋 This analysis will run 8 advanced overfitting detection methods:
   1. Data Leakage Detection
   2. Time Series Cross-Validation (3 strategies)
   3. Walk-Forward Validation
   4. Nested Cross-Validation
   5. Bootstrap Validation
   6. Learning Curves Analysis
   7. Advanced Residual Analysis
   8. Feature Importance Stability

⏱️  This may take 10-15 minutes to complete...


🎯 COMPREHENSIVE OVERFITTING ASSESSMENT REPORT
📊 Analyzing 6 models with 57834 samples and 15 features
🔍 ADVANCED METHOD 1: DATA LEAKAGE DETECTION
📊 FEATURE-TARGET CORRELATIONS:
🚨 hourly_volume_lag1: 0.9951 (p=0.0000) - POTENTIAL LEAKAGE!
🚨 hourly_volume_lag24: 0.9858 (p=0.0000) - POTENTIAL LEAKAGE!
🚨 hourly_volume_rolling_mean_24h: 0.9774 (p=0.0000) - POTENTIAL LEAKAGE!
🚨 hourly_volume_rolling_mean_7d: 0.9638 (p=0.0000) - POTENTIAL LEAKAGE!
⚠️  hourly_volume_lag168: 0.9486 (p=0.0000) - High correlation
✅ pressure: 0.7868 (p=0.0000)
✅ pressure_diff: 0.6481 (p=0.0000)
✅ mont